In [8]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont


import mss

In [2]:
def on_exists(fname: str) -> None:
    """Callback example when we try to overwrite an existing screenshot."""
    file = Path(fname)
    if file.is_file():
        newfile = file.with_name(f"{file.name}.old")
        print(f"{fname} → {newfile}")
        file.rename(newfile)

In [ ]:
def draw_grid_on_image(image, step_size, grid_color, output_path):
    """
    Draws a grid on an image using Pillow using A1 notation (Excel-style).

    Args:
        image (PIL.Image): The input image object.
        step_size (int): The distance in pixels between grid lines.
        grid_color (tuple or str): The color of the grid lines.
        output_path (str): The path to save the output image with the grid.
    """

    def get_column_label(n):
        """Converts 0-based index to Excel-style column label (A, B, ... Z, AA...)."""
        res = ""
        while n >= 0:
            res = chr(n % 26 + 65) + res
            n = n // 26 - 1
        return res

    text_color = (255, 255, 255) # White text
    box_color = (0, 0, 0) # Black box background
    
    font = ImageFont.load_default() 
    
    try:
        draw = ImageDraw.Draw(image)
        width, height = image.size

        # Draw vertical lines
        for x in range(0, width, step_size):
            line = ((x, 0), (x, height))
            draw.line(line, fill=grid_color, width=1)

        # Draw horizontal lines
        for y in range(0, height, step_size):
            line = ((0, y), (width, y))
            draw.line(line, fill=grid_color, width=1)

        # Iterate over grid cells to draw labels
        for row_idx, y in enumerate(range(0, height, step_size)):
            for col_idx, x in enumerate(range(0, width, step_size)):
                
                col_str = get_column_label(col_idx)
                label = f"{col_str}{row_idx + 1}"

                # Position text in the top-left of the square with a small padding
                text_pos = (x + 2, y + 2)
        
                # Draw a bounding box for high contrast
                bbox = draw.textbbox(text_pos, label, font=font)
                draw.rectangle(bbox, fill=box_color)
        
                # Draw the text
                draw.text(text_pos, label, fill=text_color, font=font)

        del draw

        # Save the image
        image.save(output_path)
        # Display the image
        image.show()

    except IOError as e:
        print(f"Error opening or saving image: {e}")

In [10]:
with mss.mss() as sct:
    # Get rid of the first, as it represents the "All in One" monitor:
    for num, monitor in enumerate(sct.monitors[1:], 1):
        # Get raw pixels from the screen
        sct_img = sct.grab(monitor)

        # Create the Image
        img = Image.frombytes("RGB", sct_img.size, sct_img.bgra, "raw", "BGRX")
        # The same, but less efficient:
        # img = Image.frombytes('RGB', sct_img.size, sct_img.rgb)

        # And save it!
        draw_grid_on_image(img, 50, (128, 128, 128), 'output_grid.jpg')
        